#### **Installations, Configurations, Imports, Data Preapration, Model Setup**

In [1]:
import sys
import subprocess
import importlib.util

REQUIRED_PACKAGES = {
    "torch": "torch",
    "transformers": "transformers<5",
    "peft": "peft",
    "accelerate": "accelerate",
    "datasets": "datasets",
    "huggingface_hub": "huggingface_hub",
    "pandas": "pandas",
    "numpy": "numpy",
    "tqdm": "tqdm",
    "sentence_transformers": "sentence-transformers",
    "sentencepiece": "sentencepiece",
    "sacrebleu": "sacrebleu[sentencepiece]",
}

missing = [
    pip_name
    for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing:
    print("Installing:", missing)

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        *missing,
    ])
else:
    print("All required packages are installed.")

print("Python:", sys.version)
print("Executable:", sys.executable)

All required packages are installed.
Python: 3.11.15 (main, Jun 11 2026, 15:20:16) [GCC 14.3.0]
Executable: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/bin/python


In [2]:
import os
import gc
import re
import json
import time
import random
import hashlib
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from IPython.display import display, FileLink

from datasets import (
    load_dataset,
    get_dataset_config_names,
    get_dataset_split_names,
)

from huggingface_hub import HfApi
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from peft import PeftModel


os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 3407

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.set_float32_matmul_precision("high")


# ============================================================
# Project paths
# ============================================================

PROJECT_DIR = Path(
    os.environ.get(
        "AXMT_HOME",
        str(
            Path.home()
            / "alexandriax_mt_14d"
        ),
    )
).expanduser()

BASE_MODEL_DIR = (
    PROJECT_DIR
    / "models"
    / "hf"
    / "NileChat-3B-Base"
)

EXPERIMENT_NAME = (
    "nilechat3b_alexandria_all14_"
    "context3_complete2shot_all_group_"
    "r16_alpha32_3epochs_beam4_"
    "nonquant_server5090_v1"
)

TRAINING_RUN_DIR = (
    PROJECT_DIR
    / "runs"
    / "nilechat3b_all14"
    / EXPERIMENT_NAME
)

CHECKPOINT_PATHS = {
    step: (
        TRAINING_RUN_DIR
        / f"checkpoint-{step}"
    )
    for step in [
        16000,
        16500,
        16600,
    ]
}

OUTPUT_DIR = (
    PROJECT_DIR
    / "inference_variants"
    / "92_official_private_test_system92_router_v2"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CACHE_DIR = (
    PROJECT_DIR
    / "inference_variants"
    / "_shared_cache"
    / "official_test_system92_v1"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Dataset and generation settings
# ============================================================

DATASET_NAME = "UBC-NLP/alexandria"

MAX_CONTEXT_TURNS = 3
MAX_SEQ_LENGTH = 2048
MAX_NEW_TOKENS = 120

N_FEW_SHOTS = 2
MAX_FEW_SHOT_EXAMPLE_CHARS = 450

RETRIEVER_MODEL_NAME = (
    "sentence-transformers/"
    "all-MiniLM-L6-v2"
)

RETRIEVER_BATCH_SIZE = 256

GEN_BATCH_SIZE = 2
SAVE_EVERY = 100

GENERATION_KWARGS = {
    "do_sample": False,
    "num_beams": 4,
    "num_return_sequences": 1,
    "length_penalty": 1.0,
    "early_stopping": True,
    "repetition_penalty": 1.05,
    "use_cache": True,
}

DECODE_TAG = "beam4"
PROMPT_VERSION = (
    "system92_official_test_router_v1"
)

SYSTEM_MARKER = "### System:"
INSTRUCTION_MARKER = "### Instruction:"
RESPONSE_MARKER = "### Arabic translation:"

SYSTEM_PROMPT = (
    "You are a professional machine translation system. "
    "Translate the current English dialogue turn into natural "
    "dialectal Arabic. Use the provided training examples only "
    "as style and dialect guidance. Return only the translation, "
    "without explanation."
)


# ============================================================
# Original System 92 routing
# ============================================================

SYSTEM92_ROUTES = {
    "EG": {
        "checkpoint": 16600,
        "variant": (
            "05_retrieved_two_shot_"
            "with_participants"
        ),
        "retrieval_configs": ["EG"],
        "use_participants": True,
        "selection_source": "system92_dev",
    },
    "JO": {
        "checkpoint": 16600,
        "variant": "03_retrieved_two_shot",
        "retrieval_configs": ["JO"],
        "use_participants": False,
        "selection_source": "system92_dev",
    },
    "LB": {
        "checkpoint": 16000,
        "variant": (
            "07_ckpt16000_"
            "retrieved_two_shot"
        ),
        "retrieval_configs": ["LB"],
        "use_participants": False,
        "selection_source": "system92_dev",
    },
    "MA": {
        "checkpoint": 16600,
        "variant": (
            "05_retrieved_two_shot_"
            "with_participants"
        ),
        "retrieval_configs": ["MA"],
        "use_participants": True,
        "selection_source": "system92_dev",
    },
    "MR": {
        "checkpoint": 16000,
        "variant": (
            "07_ckpt16000_"
            "retrieved_two_shot"
        ),
        "retrieval_configs": ["MR"],
        "use_participants": False,
        "selection_source": "system92_dev",
    },
    "OM": {
        "checkpoint": 16600,
        "variant": (
            "05_retrieved_two_shot_"
            "with_participants"
        ),
        "retrieval_configs": ["OM"],
        "use_participants": True,
        "selection_source": "system92_dev",
    },
    "PS": {
        "checkpoint": 16500,
        "variant": (
            "06_ckpt16500_"
            "retrieved_two_shot"
        ),
        "retrieval_configs": ["PS"],
        "use_participants": False,
        "selection_source": "system92_dev",
    },
    "SA": {
        "checkpoint": 16500,
        "variant": (
            "06_ckpt16500_"
            "retrieved_two_shot"
        ),
        "retrieval_configs": ["SA"],
        "use_participants": False,
        "selection_source": "system92_dev",
    },
    "SY": {
        "checkpoint": 16600,
        "variant": (
            "05_retrieved_two_shot_"
            "with_participants"
        ),
        "retrieval_configs": ["SY"],
        "use_participants": True,
        "selection_source": "system92_dev",
    },
    "TN": {
        "checkpoint": 16000,
        "variant": (
            "07_ckpt16000_"
            "retrieved_two_shot"
        ),
        "retrieval_configs": ["TN"],
        "use_participants": False,
        "selection_source": "system92_dev",
    },
    "YE": {
        "checkpoint": 16500,
        "variant": (
            "06_ckpt16500_"
            "retrieved_two_shot"
        ),
        "retrieval_configs": ["YE"],
        "use_participants": False,
        "selection_source": "system92_dev",
    },
}


# ============================================================
# Explicit zero-shot fallbacks
# ============================================================

UNSEEN_ROUTES = {
    # Libya -> geographically closest available TN expert.
    "LY": {
        "checkpoint": 16000,
        "variant": (
            "92_unseen_LY_proxy_TN"
        ),
        "retrieval_configs": ["TN"],
        "use_participants": False,
        "selection_source": (
            "unseen_geographic_proxy_TN"
        ),
    },

    # Sudan -> Nile Valley proxy through EG.
    "SD": {
        "checkpoint": 16600,
        "variant": (
            "92_unseen_SD_proxy_EG"
        ),
        "retrieval_configs": ["EG"],
        "use_participants": True,
        "selection_source": (
            "unseen_geographic_proxy_EG"
        ),
    },
}

ALL_ROUTES = {
    **SYSTEM92_ROUTES,
    **UNSEEN_ROUTES,
}


# ============================================================
# Safety validation
# ============================================================

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Project directory not found: {PROJECT_DIR}"
    )

if not BASE_MODEL_DIR.exists():
    raise FileNotFoundError(
        f"Base model not found: {BASE_MODEL_DIR}"
    )

for step, checkpoint_path in CHECKPOINT_PATHS.items():
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Checkpoint {step} not found: "
            f"{checkpoint_path}"
        )

    if not (
        checkpoint_path
        / "adapter_config.json"
    ).exists():
        raise FileNotFoundError(
            f"adapter_config.json missing: "
            f"{checkpoint_path}"
        )

print("Project:", PROJECT_DIR)
print("Base model:", BASE_MODEL_DIR)
print("Output:", OUTPUT_DIR)
print("Save every:", SAVE_EVERY)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "BF16:",
        torch.cuda.is_bf16_supported(),
    )

display(
    pd.DataFrame([
        {
            "config": config,
            **route,
        }
        for config, route
        in ALL_ROUTES.items()
    ])
)

Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /home/mabdallah/alexandriax_mt_14d/envs/axmt_py311/lib/python3.11/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


Project: /home/mabdallah/alexandriax_mt_14d
Base model: /home/mabdallah/alexandriax_mt_14d/models/hf/NileChat-3B-Base
Output: /home/mabdallah/alexandriax_mt_14d/inference_variants/92_official_private_test_system92_router_v2
Save every: 100
CUDA: True
GPU: NVIDIA GeForce RTX 5090
BF16: True


,config,checkpoint,variant,retrieval_configs,use_participants,selection_source
0,EG,16600,05_retrieved_two_shot_with_participants,[EG],True,system92_dev
1,JO,16600,03_retrieved_two_shot,[JO],False,system92_dev
2,LB,16000,07_ckpt16000_retrieved_two_shot,[LB],False,system92_dev
3,MA,16600,05_retrieved_two_shot_with_participants,[MA],True,system92_dev
4,MR,16000,07_ckpt16000_retrieved_two_shot,[MR],False,system92_dev
5,OM,16600,05_retrieved_two_shot_with_participants,[OM],True,system92_dev
6,PS,16500,06_ckpt16500_retrieved_two_shot,[PS],False,system92_dev
7,SA,16500,06_ckpt16500_retrieved_two_shot,[SA],False,system92_dev
8,SY,16600,05_retrieved_two_shot_with_participants,[SY],True,system92_dev
9,TN,16000,07_ckpt16000_retrieved_two_shot,[TN],False,system92_dev


In [4]:
# ============================================================
# Helpers
# ============================================================

def to_plain(value):
    if isinstance(value, np.ndarray):
        return [to_plain(item) for item in value.tolist()]

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, tuple):
        return [to_plain(item) for item in value]

    if isinstance(value, list):
        return [to_plain(item) for item in value]

    if isinstance(value, dict):
        return {
            str(key): to_plain(item)
            for key, item in value.items()
        }

    return value


def scalar_text(value):
    value = to_plain(value)

    if value is None:
        return ""

    if isinstance(value, (list, dict)):
        return str(value).strip()

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    return str(value).strip()


def normalize_turn_list(value):
    value = to_plain(value)

    if value is None:
        return []

    if isinstance(value, list):
        normalized = []

        for item in value:
            item = to_plain(item)

            if isinstance(item, dict):
                normalized.append(item)
            elif item is not None:
                normalized.append({"text": scalar_text(item)})

        return normalized

    if isinstance(value, dict):
        lengths = [
            len(to_plain(item))
            for item in value.values()
            if isinstance(to_plain(item), list)
        ]

        if not lengths:
            return []

        rows = []

        for row_index in range(max(lengths)):
            item = {}

            for key, values in value.items():
                values = to_plain(values)

                if isinstance(values, list):
                    item[key] = (
                        values[row_index]
                        if row_index < len(values)
                        else None
                    )
                else:
                    item[key] = values

            rows.append(item)

        return rows

    return []


def turn_field(turn, possible_keys, default=""):
    turn = to_plain(turn)

    if not isinstance(turn, dict):
        return default

    for key in possible_keys:
        if key not in turn:
            continue

        value = scalar_text(turn[key])

        if value:
            return value

    return default


def turn_text(turn):
    return turn_field(
        turn,
        [
            "text",
            "source",
            "source_text",
            "english",
            "english_text",
            "sentence",
            "utterance",
            "content",
            "value"
        ],
        default=""
    )


def extract_turn_order(turn, fallback_index):
    raw_value = turn_field(
        turn,
        ["turn_order", "turn_id", "order", "idx", "index"],
        default=""
    )

    if raw_value:
        try:
            return int(raw_value)
        except Exception:
            pass

    return int(fallback_index + 1)


def dataframe_id_hash(frame):
    hasher = hashlib.sha256()

    for source_id in frame["source_id"].astype(str):
        hasher.update(source_id.encode("utf-8"))
        hasher.update(b"\n")

    return hasher.hexdigest()


# ============================================================
# Flatten private source-only test data
# ============================================================

def flatten_test_split(dataset, config_name):
    records = []

    for conversation_index, raw_row in enumerate(dataset):
        row = to_plain(dict(raw_row))

        conversation_id = scalar_text(
            row.get(
                "conv_id",
                row.get(
                    "conversation_id",
                    f"{config_name}_test_{conversation_index}"
                )
            )
        )

        country = (
            scalar_text(row.get("country", config_name))
            or config_name
        )

        dialect = scalar_text(row.get("dialect", ""))
        domain = scalar_text(row.get("domain", ""))
        participants = scalar_text(row.get("participants", ""))

        english_turns = normalize_turn_list(
            row.get("english_conversation", [])
        )

        if not english_turns:
            raise RuntimeError(
                f"No English turns in {config_name}/{conversation_id}"
            )

        for turn_index, english_turn in enumerate(english_turns):
            source_text = turn_text(english_turn)

            if not source_text:
                raise RuntimeError(
                    f"Empty source: {config_name}/"
                    f"{conversation_id}/{turn_index + 1}"
                )

            turn_order = extract_turn_order(
                english_turn,
                turn_index
            )

            speaker = turn_field(
                english_turn,
                ["speaker", "role", "speaker_role", "participant"],
                default=""
            )

            gender_direction = turn_field(
                english_turn,
                [
                    "direction",
                    "gender_direction",
                    "speaker_addressee_gender"
                ],
                default=""
            )

            previous_turns = []
            previous_start = max(
                0,
                turn_index - MAX_CONTEXT_TURNS
            )

            for previous_index in range(
                previous_start,
                turn_index
            ):
                previous_turn = english_turns[previous_index]

                previous_turns.append({
                    "turn_order": extract_turn_order(
                        previous_turn,
                        previous_index
                    ),
                    "speaker": turn_field(
                        previous_turn,
                        [
                            "speaker",
                            "role",
                            "speaker_role",
                            "participant"
                        ],
                        default=""
                    ),
                    "direction": turn_field(
                        previous_turn,
                        [
                            "direction",
                            "gender_direction",
                            "speaker_addressee_gender"
                        ],
                        default=""
                    ),
                    "text": turn_text(previous_turn)
                })

            records.append({
                "source_id": (
                    f"{config_name}_test_"
                    f"{conversation_id}_{turn_order}"
                ),
                "config": config_name,
                "country": country,
                "split": "test",
                "conversation_id": conversation_id,
                "turn_order": int(turn_order),
                "turn_id": int(turn_order),
                "dialect": dialect,
                "domain": domain,
                "participants": participants,
                "speaker": speaker,
                "gender_direction": gender_direction,
                "previous_english_turns": previous_turns,
                "source_text": source_text
            })

    return records


# ============================================================
# Load official Codabench private-test archive
# ============================================================

PRIVATE_TEST_ARCHIVE = (
    PROJECT_DIR
    / "data"
    / "nilechat3b_all14"
    / "alexandriax_private_test.zip"
)
PRIVATE_TEST_EXTRACT_DIR = (
    CACHE_DIR
    / "codabench_private_test_807229"
)

EXPECTED_TEST_TURNS = 14459
DATASET_REVISION = "codabench_private_test_807229"

if not PRIVATE_TEST_ARCHIVE.exists():
    raise FileNotFoundError(
        "Download Test/Public Data from the Codabench Files page "
        f"and place it at: {PRIVATE_TEST_ARCHIVE}"
    )

PRIVATE_TEST_EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

with zipfile.ZipFile(PRIVATE_TEST_ARCHIVE) as archive:
    archive.extractall(PRIVATE_TEST_EXTRACT_DIR)

    print("Files in private-test archive:")

    for name in archive.namelist():
        print(name)

source_files = sorted([
    path
    for path in PRIVATE_TEST_EXTRACT_DIR.rglob("*")
    if path.suffix.lower() in {".jsonl", ".json"}
    and "__MACOSX" not in path.parts
])

if not source_files:
    raise RuntimeError(
        "No JSON or JSONL files were found in the "
        "Codabench private-test archive."
    )


def read_private_records(path):
    if path.suffix.lower() == ".jsonl":
        records = []

        with open(path, encoding="utf-8") as file:
            for line in file:
                line = line.strip()

                if line:
                    records.append(json.loads(line))

        return records

    with open(path, encoding="utf-8") as file:
        payload = json.load(file)

    if isinstance(payload, list):
        return payload

    if isinstance(payload, dict):
        for key in [
            "data",
            "records",
            "conversations",
            "examples"
        ]:
            if isinstance(payload.get(key), list):
                return payload[key]

        if (
            "conv_id" in payload
            or "conversation_id" in payload
        ):
            return [payload]

        combined = []

        for value in payload.values():
            if isinstance(value, list):
                combined.extend(value)

        if combined:
            return combined

    return []


private_conversations = []

for source_file in source_files:
    file_name = source_file.name.upper()

    config_match = re.search(
        r"(?:^|[_\-.])"
        r"(EG|JO|LB|LY|MA|MR|OM|PS|SA|SD|SY|TN|YE)"
        r"(?:[_\-.]|$)",
        file_name
    )

    config_hint = (
        config_match.group(1)
        if config_match
        else ""
    )

    records = read_private_records(source_file)

    print(
        source_file.relative_to(PRIVATE_TEST_EXTRACT_DIR),
        "records:",
        len(records)
    )

    for record in records:
        record = to_plain(record)

        if not isinstance(record, dict):
            continue

        if "english_conversation" not in record:
            for alternative in [
                "turns",
                "dialogue",
                "conversation",
                "source_conversation"
            ]:
                if isinstance(record.get(alternative), list):
                    record["english_conversation"] = record[alternative]
                    break

        config_name = scalar_text(
            record.get(
                "country",
                record.get("config", config_hint)
            )
        ).upper()

        if config_name not in ALL_ROUTES and config_hint:
            config_name = config_hint

        if config_name not in ALL_ROUTES:
            raise RuntimeError(
                f"Unknown config {config_name!r} in {source_file}"
            )

        record["country"] = config_name

        private_conversations.append(
            (config_name, record)
        )

print(
    "Private-test conversations read:",
    len(private_conversations)
)

test_records = []

for config_name, record in private_conversations:
    test_records.extend(
        flatten_test_split(
            [record],
            config_name
        )
    )

official_test_df = pd.DataFrame(test_records)

official_test_df = (
    official_test_df
    .sort_values([
        "config",
        "conversation_id",
        "turn_order"
    ])
    .reset_index(drop=True)
)

official_test_df["source_id"] = (
    official_test_df["source_id"].astype(str)
)

official_test_df["config"] = (
    official_test_df["config"].astype(str)
)

official_test_df["conversation_id"] = (
    official_test_df["conversation_id"].astype(str)
)

official_test_df["turn_order"] = (
    pd.to_numeric(
        official_test_df["turn_order"],
        errors="raise"
    ).astype(int)
)

official_test_df["source_text"] = (
    official_test_df["source_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

official_test_df["_test_row_idx"] = np.arange(
    len(official_test_df),
    dtype=np.int64
)

if len(official_test_df) != EXPECTED_TEST_TURNS:
    raise RuntimeError(
        f"Wrong test archive: expected {EXPECTED_TEST_TURNS} turns, "
        f"found {len(official_test_df)}."
    )

if official_test_df["source_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate source_id in the private test."
    )

if (
    official_test_df[
        ["config", "conversation_id", "turn_order"]
    ]
    .duplicated()
    .any()
):
    raise RuntimeError(
        "Duplicate official private-test alignment key."
    )

if (official_test_df["source_text"] == "").any():
    raise RuntimeError(
        "Empty English source in private test."
    )

expected_scorer_keys = {
    ("EG", "B0-1-0-132", 1),
    ("EG", "B0-1-0-132", 2),
    ("EG", "B0-1-0-182", 1),
    ("EG", "B0-1-0-182", 2),
    ("EG", "B0-1-0-182", 3)
}

actual_keys = set(zip(
    official_test_df["config"],
    official_test_df["conversation_id"],
    official_test_df["turn_order"]
))

missing_scorer_keys = expected_scorer_keys - actual_keys

if missing_scorer_keys:
    raise RuntimeError(
        "This archive does not match the scorer. "
        f"Missing known keys: {sorted(missing_scorer_keys)}"
    )

TEST_CONFIGS = sorted(
    official_test_df["config"].unique()
)

TEST_ID_HASH = dataframe_id_hash(
    official_test_df
)

# Use the already prepared training pool.
LEGACY_TRAIN_CACHE_PATH = (
    PROJECT_DIR
    / "inference_variants"
    / "_shared_cache"
    / "paired_dev_train_v3"
    / "train_fewshot_pool_df.pkl"
)

if not LEGACY_TRAIN_CACHE_PATH.exists():
    raise FileNotFoundError(
        f"Missing training pool: {LEGACY_TRAIN_CACHE_PATH}"
    )

train_fewshot_df = pd.read_pickle(
    LEGACY_TRAIN_CACHE_PATH
)

TRAIN_CONFIGS = sorted(
    train_fewshot_df[
        "config"
    ].astype(str).unique()
)

UNSEEN_TEST_CONFIGS = sorted(
    set(TEST_CONFIGS) - set(TRAIN_CONFIGS)
)

print()
print("OFFICIAL PRIVATE TEST LOADED")
print("Turns:", len(official_test_df))
print(
    "Conversations:",
    official_test_df[
        ["config", "conversation_id"]
    ].drop_duplicates().shape[0]
)
print("Configs:", TEST_CONFIGS)
print("Unseen configs:", UNSEEN_TEST_CONFIGS)
print("Test ID hash:", TEST_ID_HASH)

display(
    official_test_df
    .groupby("config", as_index=False)
    .agg(
        conversations=("conversation_id", "nunique"),
        turns=("source_id", "size")
    )
)

Files in private-test archive:
alexandria_MA_private_test_input.jsonl
alexandria_LY_private_test_input.jsonl
alexandria_SY_private_test_input.jsonl
alexandria_EG_private_test_input.jsonl
alexandria_TN_private_test_input.jsonl
alexandria_PS_private_test_input.jsonl
alexandria_LB_private_test_input.jsonl
alexandria_YE_private_test_input.jsonl
manifest.json
alexandria_MR_private_test_input.jsonl
alexandria_SA_private_test_input.jsonl
alexandria_SD_private_test_input.jsonl
alexandria_OM_private_test_input.jsonl
alexandria_JO_private_test_input.jsonl
alexandria_EG_private_test_input.jsonl records: 359
alexandria_JO_private_test_input.jsonl records: 348
alexandria_LB_private_test_input.jsonl records: 368
alexandria_LY_private_test_input.jsonl records: 423
alexandria_MA_private_test_input.jsonl records: 362
alexandria_MR_private_test_input.jsonl records: 365
alexandria_OM_private_test_input.jsonl records: 370
alexandria_PS_private_test_input.jsonl records: 351
alexandria_SA_private_test_input

,config,conversations,turns
0,EG,359,1113
1,JO,348,1109
2,LB,368,1110
3,LY,423,1309
4,MA,362,1111
5,MR,365,1119
6,OM,370,1107
7,PS,351,1111
8,SA,359,1114
9,SD,283,915


#### **Select retrieved examples and assign routes**

In [5]:
# ============================================================
# Atomic persistence helpers
# ============================================================

def atomic_json_save(payload, path):
    path = Path(path)

    temporary_path = Path(
        str(path) + ".tmp"
    )

    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            payload,
            file,
            ensure_ascii=False,
            indent=2,
        )

    os.replace(
        temporary_path,
        path,
    )


def atomic_pickle_save(frame, path):
    path = Path(path)

    temporary_path = Path(
        str(path) + ".tmp"
    )

    frame.to_pickle(
        temporary_path
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_npy_save(array, path):
    path = Path(path)

    temporary_path = Path(
        str(path) + ".tmp"
    )

    with open(
        temporary_path,
        "wb",
    ) as file:
        np.save(
            file,
            array,
        )

    os.replace(
        temporary_path,
        path,
    )


# ============================================================
# Prepare training retrieval pool
# ============================================================

FEW_SHOT_COLUMNS = [
    "source_id",
    "config",
    "conversation_id",
    "dialect",
    "domain",
    "participants",
    "speaker",
    "gender_direction",
    "source_text",
    "target_arabic",
    "translator_id",
    "reviewer_id",
]

missing_columns = [
    column
    for column in FEW_SHOT_COLUMNS
    if column not in train_fewshot_df.columns
]

if missing_columns:
    raise RuntimeError(
        "Training retrieval pool is missing: "
        f"{missing_columns}"
    )

train_shot_pool = (
    train_fewshot_df[
        FEW_SHOT_COLUMNS
    ]
    .copy()
    .reset_index(drop=True)
)

train_shot_pool["fewshot_total_chars"] = (
    train_shot_pool[
        "source_text"
    ].astype(str).str.len()
    + train_shot_pool[
        "target_arabic"
    ].astype(str).str.len()
)

retrieval_train_pool = (
    train_shot_pool[
        train_shot_pool[
            "fewshot_total_chars"
        ]
        <= MAX_FEW_SHOT_EXAMPLE_CHARS
    ]
    .copy()
    .reset_index(drop=True)
)

if len(retrieval_train_pool) == 0:
    raise RuntimeError(
        "The retrieval pool is empty."
    )

indices_by_config = {
    config: group.index.to_numpy(
        dtype=np.int64
    )
    for config, group
    in retrieval_train_pool.groupby(
        "config",
        sort=False,
    )
}

indices_by_config_domain = {
    key: group.index.to_numpy(
        dtype=np.int64
    )
    for key, group
    in retrieval_train_pool.groupby(
        [
            "config",
            "domain",
        ],
        sort=False,
    )
}


# ============================================================
# Attach System 92 routes
# ============================================================

inference_test_df = (
    official_test_df.copy()
)

inference_test_df[
    "route_checkpoint"
] = inference_test_df[
    "config"
].map(
    lambda config: int(
        ALL_ROUTES[config][
            "checkpoint"
        ]
    )
)

inference_test_df[
    "route_variant"
] = inference_test_df[
    "config"
].map(
    lambda config: (
        ALL_ROUTES[config][
            "variant"
        ]
    )
)

inference_test_df[
    "route_use_participants"
] = inference_test_df[
    "config"
].map(
    lambda config: bool(
        ALL_ROUTES[config][
            "use_participants"
        ]
    )
)

inference_test_df[
    "route_retrieval_configs"
] = inference_test_df[
    "config"
].map(
    lambda config: list(
        ALL_ROUTES[config][
            "retrieval_configs"
        ]
    )
)

inference_test_df[
    "route_selection_source"
] = inference_test_df[
    "config"
].map(
    lambda config: (
        ALL_ROUTES[config][
            "selection_source"
        ]
    )
)


# ============================================================
# Build or load semantic embeddings
# ============================================================

EMBEDDING_CACHE_DIR = (
    CACHE_DIR
    / "semantic_retrieval"
)

EMBEDDING_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TRAIN_EMBEDDINGS_PATH = (
    EMBEDDING_CACHE_DIR
    / "train_embeddings.npy"
)

TEST_EMBEDDINGS_PATH = (
    EMBEDDING_CACHE_DIR
    / "test_embeddings.npy"
)

EMBEDDING_MANIFEST_PATH = (
    EMBEDDING_CACHE_DIR
    / "manifest.json"
)

TRAIN_ID_HASH = dataframe_id_hash(
    retrieval_train_pool
)

expected_embedding_manifest = {
    "dataset_revision": DATASET_REVISION,
    "retriever_model": (
        RETRIEVER_MODEL_NAME
    ),
    "train_rows": int(
        len(retrieval_train_pool)
    ),
    "test_rows": int(
        len(inference_test_df)
    ),
    "train_hash": TRAIN_ID_HASH,
    "test_hash": TEST_ID_HASH,
}

embedding_cache_valid = False

if (
    EMBEDDING_MANIFEST_PATH.exists()
    and TRAIN_EMBEDDINGS_PATH.exists()
    and TEST_EMBEDDINGS_PATH.exists()
):
    try:
        with open(
            EMBEDDING_MANIFEST_PATH,
            "r",
            encoding="utf-8",
        ) as file:
            existing_manifest = json.load(
                file
            )

        cached_train_embeddings = np.load(
            TRAIN_EMBEDDINGS_PATH,
            mmap_mode="r",
        )

        cached_test_embeddings = np.load(
            TEST_EMBEDDINGS_PATH,
            mmap_mode="r",
        )

        embedding_cache_valid = (
            existing_manifest
            == expected_embedding_manifest
            and cached_train_embeddings.shape[0]
            == len(retrieval_train_pool)
            and cached_test_embeddings.shape[0]
            == len(inference_test_df)
        )

        del cached_train_embeddings
        del cached_test_embeddings

    except Exception as error:
        print(
            "Ignoring invalid embedding cache:",
            repr(error),
        )

if not embedding_cache_valid:
    print(
        "Building train/test semantic embeddings..."
    )

    from sentence_transformers import (
        SentenceTransformer
    )

    retrieval_device = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    retrieval_model = SentenceTransformer(
        RETRIEVER_MODEL_NAME,
        device=retrieval_device,
    )

    train_embeddings_array = (
        retrieval_model.encode(
            retrieval_train_pool[
                "source_text"
            ].fillna("").astype(str).tolist(),
            batch_size=RETRIEVER_BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype(np.float32)
    )

    test_embeddings_array = (
        retrieval_model.encode(
            inference_test_df[
                "source_text"
            ].fillna("").astype(str).tolist(),
            batch_size=RETRIEVER_BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype(np.float32)
    )

    atomic_npy_save(
        train_embeddings_array,
        TRAIN_EMBEDDINGS_PATH,
    )

    atomic_npy_save(
        test_embeddings_array,
        TEST_EMBEDDINGS_PATH,
    )

    atomic_json_save(
        expected_embedding_manifest,
        EMBEDDING_MANIFEST_PATH,
    )

    del retrieval_model
    del train_embeddings_array
    del test_embeddings_array

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

train_embeddings = np.load(
    TRAIN_EMBEDDINGS_PATH,
    mmap_mode="r",
)

test_embeddings = np.load(
    TEST_EMBEDDINGS_PATH,
    mmap_mode="r",
)

print(
    "Train embeddings:",
    train_embeddings.shape,
)

print(
    "Test embeddings:",
    test_embeddings.shape,
)


# ============================================================
# Retrieve two examples for each test turn
# ============================================================

def select_test_two_shots(
    row,
    number_of_shots=N_FEW_SHOTS,
):
    donor_configs = list(
        row[
            "route_retrieval_configs"
        ]
    )

    domain = scalar_text(
        row.get("domain", "")
    )

    candidate_parts = []

    for donor_config in donor_configs:
        indices = (
            indices_by_config_domain.get(
                (
                    donor_config,
                    domain,
                )
            )
        )

        if indices is not None:
            candidate_parts.append(
                indices
            )

    if (
        not candidate_parts
        or sum(
            len(part)
            for part in candidate_parts
        ) < number_of_shots
    ):
        candidate_parts = [
            indices_by_config[donor]
            for donor in donor_configs
            if donor in indices_by_config
        ]

    if not candidate_parts:
        raise RuntimeError(
            "No retrieval candidates for "
            f"{row['config']} using "
            f"{donor_configs}"
        )

    candidate_indices = np.unique(
        np.concatenate(
            candidate_parts
        )
    )

    query_index = int(
        row["_test_row_idx"]
    )

    query_embedding = np.asarray(
        test_embeddings[
            query_index
        ],
        dtype=np.float32,
    )

    candidate_embedding_matrix = (
        np.asarray(
            train_embeddings[
                candidate_indices
            ],
            dtype=np.float32,
        )
    )

    scores = (
        candidate_embedding_matrix
        @ query_embedding
    )

    candidate_metadata = (
        retrieval_train_pool.iloc[
            candidate_indices
        ]
    )

    target_dialect = scalar_text(
        row.get("dialect", "")
    )

    direction = scalar_text(
        row.get(
            "gender_direction",
            "",
        )
    )

    speaker = scalar_text(
        row.get("speaker", "")
    )

    if target_dialect:
        scores += (
            candidate_metadata[
                "dialect"
            ].astype(str).to_numpy()
            == target_dialect
        ).astype(np.float32) * 0.05

    if direction:
        scores += (
            candidate_metadata[
                "gender_direction"
            ].astype(str).to_numpy()
            == direction
        ).astype(np.float32) * 0.03

    if speaker:
        scores += (
            candidate_metadata[
                "speaker"
            ].astype(str).to_numpy()
            == speaker
        ).astype(np.float32) * 0.02

    ranked_positions = np.argsort(
        -scores
    )

    selected = []
    used_conversations = set()
    used_source_ids = set()

    # Prefer separate conversations.
    for position in ranked_positions:
        pool_index = int(
            candidate_indices[
                position
            ]
        )

        candidate = (
            retrieval_train_pool.iloc[
                pool_index
            ]
        )

        conversation_key = (
            scalar_text(
                candidate["config"]
            ),
            scalar_text(
                candidate[
                    "conversation_id"
                ]
            ),
        )

        if (
            conversation_key
            in used_conversations
        ):
            continue

        record = candidate[
            FEW_SHOT_COLUMNS
        ].to_dict()

        selected.append(record)

        used_conversations.add(
            conversation_key
        )

        used_source_ids.add(
            scalar_text(
                candidate["source_id"]
            )
        )

        if (
            len(selected)
            >= number_of_shots
        ):
            break

    # Fallback if unique conversations were insufficient.
    if len(selected) < number_of_shots:
        for position in ranked_positions:
            pool_index = int(
                candidate_indices[
                    position
                ]
            )

            candidate = (
                retrieval_train_pool.iloc[
                    pool_index
                ]
            )

            source_id = scalar_text(
                candidate["source_id"]
            )

            if source_id in used_source_ids:
                continue

            selected.append(
                candidate[
                    FEW_SHOT_COLUMNS
                ].to_dict()
            )

            used_source_ids.add(
                source_id
            )

            if (
                len(selected)
                >= number_of_shots
            ):
                break

    if len(selected) < number_of_shots:
        raise RuntimeError(
            "Could not select enough examples "
            f"for {row['source_id']}"
        )

    return selected[
        :number_of_shots
    ]


shot_payload = {
    "dataset_revision": DATASET_REVISION,
    "test_hash": TEST_ID_HASH,
    "train_hash": TRAIN_ID_HASH,
    "retriever_model": (
        RETRIEVER_MODEL_NAME
    ),
    "routes": ALL_ROUTES,
    "n_few_shots": N_FEW_SHOTS,
}

SHOT_FINGERPRINT = hashlib.sha256(
    json.dumps(
        shot_payload,
        ensure_ascii=False,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()

SHOTS_CACHE_PATH = (
    OUTPUT_DIR
    / "selected_few_shots.pkl"
)

loaded_shots = False

if SHOTS_CACHE_PATH.exists():
    cached_shots = pd.read_pickle(
        SHOTS_CACHE_PATH
    )

    if (
        "shot_fingerprint"
        in cached_shots.columns
        and set(
            cached_shots[
                "shot_fingerprint"
            ].astype(str)
        )
        == {SHOT_FINGERPRINT}
        and set(
            cached_shots[
                "source_id"
            ].astype(str)
        )
        == set(
            inference_test_df[
                "source_id"
            ].astype(str)
        )
    ):
        shot_map = dict(zip(
            cached_shots[
                "source_id"
            ].astype(str),
            cached_shots[
                "few_shot_examples"
            ],
        ))

        inference_test_df[
            "few_shot_examples"
        ] = (
            inference_test_df[
                "source_id"
            ].astype(str).map(
                shot_map
            )
        )

        loaded_shots = True

        print(
            "Loaded selected-shot cache:",
            SHOTS_CACHE_PATH,
        )

if not loaded_shots:
    selected_shots = []

    for _, row in tqdm(
        inference_test_df.iterrows(),
        total=len(inference_test_df),
        desc="Selecting test two-shots",
    ):
        selected_shots.append(
            select_test_two_shots(
                row.to_dict()
            )
        )

    inference_test_df[
        "few_shot_examples"
    ] = selected_shots

    shots_cache_df = pd.DataFrame({
        "source_id": (
            inference_test_df[
                "source_id"
            ].astype(str)
        ),
        "few_shot_examples": (
            selected_shots
        ),
        "shot_fingerprint": (
            SHOT_FINGERPRINT
        ),
    })

    atomic_pickle_save(
        shots_cache_df,
        SHOTS_CACHE_PATH,
    )

    print(
        "Saved selected-shot cache:",
        SHOTS_CACHE_PATH,
    )

del train_embeddings
del test_embeddings

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nRouting summary:")
display(
    inference_test_df
    .groupby(
        [
            "config",
            "route_checkpoint",
            "route_variant",
            "route_selection_source",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        turns=(
            "source_id",
            "size",
        )
    )
)

Building train/test semantic embeddings...


Batches:   0%|          | 0/260 [00:00<?, ?it/s]

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

Train embeddings: (66344, 384)
Test embeddings: (14459, 384)


Selecting test two-shots:   0%|          | 0/14459 [00:00<?, ?it/s]

Saved selected-shot cache: /home/mabdallah/alexandriax_mt_14d/inference_variants/92_official_private_test_system92_router_v2/selected_few_shots.pkl

Routing summary:


,config,route_checkpoint,route_variant,route_selection_source,turns
0,EG,16600,05_retrieved_two_shot_with_participants,system92_dev,1113
1,JO,16600,03_retrieved_two_shot,system92_dev,1109
2,LB,16000,07_ckpt16000_retrieved_two_shot,system92_dev,1110
3,LY,16000,92_unseen_LY_proxy_TN,unseen_geographic_proxy_TN,1309
4,MA,16600,05_retrieved_two_shot_with_participants,system92_dev,1111
5,MR,16000,07_ckpt16000_retrieved_two_shot,system92_dev,1119
6,OM,16600,05_retrieved_two_shot_with_participants,system92_dev,1107
7,PS,16500,06_ckpt16500_retrieved_two_shot,system92_dev,1111
8,SA,16500,06_ckpt16500_retrieved_two_shot,system92_dev,1114
9,SD,16600,92_unseen_SD_proxy_EG,unseen_geographic_proxy_EG,915


### **Prompt, model and generation functions**

In [6]:
def safe_string(value):
    return scalar_text(value)


def build_context(row, spec):
    previous_turns = to_plain(
        row.get(
            "previous_english_turns",
            [],
        )
    )

    if not previous_turns:
        return "No previous context."

    lines = []

    for position, previous_turn in enumerate(
        previous_turns,
        start=1,
    ):
        if isinstance(
            previous_turn,
            dict,
        ):
            text = safe_string(
                previous_turn.get(
                    "text",
                    "",
                )
            )

            speaker = safe_string(
                previous_turn.get(
                    "speaker",
                    "",
                )
            )
        else:
            text = safe_string(
                previous_turn
            )

            speaker = ""

        if not text:
            continue

        if (
            spec[
                "use_previous_speakers"
            ]
            and speaker
        ):
            lines.append(
                f"{position}. "
                f"{speaker}: {text}"
            )
        else:
            lines.append(
                f"{position}. {text}"
            )

    return (
        "\n".join(lines)
        if lines
        else "No previous context."
    )


def build_metadata(row, spec):
    fields = [
        (
            "Country/config",
            row.get("config", ""),
        ),
        (
            "Target dialect",
            row.get("dialect", ""),
        ),
        (
            "Domain",
            row.get("domain", ""),
        ),
    ]

    if spec["use_participants"]:
        fields.append((
            "Persona/Roles",
            row.get(
                "participants",
                "",
            ),
        ))

    fields.append((
        "Current speaker",
        row.get("speaker", ""),
    ))

    if spec["use_direction"]:
        fields.append((
            (
                "Speaker-to-addressee "
                "gender direction"
            ),
            row.get(
                "gender_direction",
                "",
            ),
        ))

    lines = []

    for label, value in fields:
        value = safe_string(value)

        if value:
            lines.append(
                f"{label}: {value}"
            )

    return (
        "\n".join(lines)
        if lines
        else "No metadata."
    )


def build_few_shot_block(row):
    examples = to_plain(
        row.get(
            "few_shot_examples",
            [],
        )
    )

    if not examples:
        return "No examples available."

    blocks = []

    for example_index, example in enumerate(
        examples,
        start=1,
    ):
        metadata_parts = []

        config = safe_string(
            example.get("config", "")
        )

        dialect = safe_string(
            example.get("dialect", "")
        )

        domain = safe_string(
            example.get("domain", "")
        )

        if config:
            metadata_parts.append(
                f"config={config}"
            )

        if dialect:
            metadata_parts.append(
                f"dialect={dialect}"
            )

        if domain:
            metadata_parts.append(
                f"domain={domain}"
            )

        metadata_line = (
            ", ".join(metadata_parts)
            if metadata_parts
            else "no metadata"
        )

        source = safe_string(
            example.get(
                "source_text",
                "",
            )
        )

        target = safe_string(
            example.get(
                "target_arabic",
                "",
            )
        )

        blocks.append(
            f"""Example {example_index} ({metadata_line})
English:
{source}

Arabic:
{target}"""
        )

    return "\n\n".join(blocks)


def format_generation_prompt(
    row,
    spec,
):
    context = build_context(
        row,
        spec,
    )

    metadata = build_metadata(
        row,
        spec,
    )

    examples = build_few_shot_block(
        row
    )

    user_prompt = f"""Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Few-shot training examples:
{examples}

Metadata:
{metadata}

Previous English dialogue context:
{context}

Current English turn:
{row["source_text"]}

Rules:
- Preserve the meaning exactly.
- Use the target local dialect, not Modern Standard Arabic unless it is natural in context.
- Follow the dialect/style pattern shown in the few-shot examples when relevant.
- Do not copy the few-shot examples.
- Preserve names, numbers, named entities, and technical terms when appropriate.
- Keep the tone appropriate for the speaker and domain.
- Return only the Arabic translation."""

    return (
        f"{SYSTEM_MARKER}\n"
        f"{SYSTEM_PROMPT}\n\n"
        f"{INSTRUCTION_MARKER}\n"
        f"{user_prompt}\n\n"
        f"{RESPONSE_MARKER}\n"
    )


def arabic_ratio(text):
    text = safe_string(text)

    if not text:
        return 0.0

    arabic_characters = sum(
        1
        for character in text
        if "\u0600"
        <= character
        <= "\u06FF"
    )

    return (
        arabic_characters
        / max(len(text), 1)
    )


def latin_ratio(text):
    text = safe_string(text)

    if not text:
        return 0.0

    latin_characters = sum(
        1
        for character in text
        if (
            "a"
            <= character.lower()
            <= "z"
        )
    )

    return (
        latin_characters
        / max(len(text), 1)
    )


def strip_special_tokens(
    text,
    tokenizer=None,
):
    text = safe_string(text)

    tokens = [
        "<|endoftext|>",
        "<|im_end|>",
        "<|im_start|>",
        "<turn|>",
    ]

    if tokenizer is not None:
        for token in [
            getattr(
                tokenizer,
                "eos_token",
                None,
            ),
            getattr(
                tokenizer,
                "pad_token",
                None,
            ),
        ]:
            if token:
                tokens.append(token)

    for token in tokens:
        text = text.replace(
            token,
            "",
        )

    return text.strip()


def postprocess_prediction(
    text,
    tokenizer=None,
):
    text = strip_special_tokens(
        text,
        tokenizer,
    )

    text = (
        text
        .replace("\r\n", "\n")
        .replace("\r", "\n")
        .strip()
    )

    split_markers = [
        "### Arabic translation:",
        "### Arabic Translation:",
        "Arabic translation:",
        "Arabic Translation:",
        "### Arabic:",
        "Arabic:",
        "الترجمة العربية:",
        "الترجمة:",
    ]

    for marker in split_markers:
        if marker in text:
            text = text.split(
                marker
            )[-1].strip()

    text = (
        text
        .strip("`")
        .replace("###", "")
        .strip()
    )

    lines = [
        line.strip()
        for line in text.split("\n")
        if line.strip()
    ]

    drop_prefixes = [
        "return only",
        "do not",
        "don't add",
        "preserve",
        "use the",
        "task:",
        "rules:",
        "metadata:",
        "previous english",
        "current english",
        "few-shot",
        "example",
        "english:",
        "arabic translation",
        "translation:",
        "target dialect",
        "speaker-to-addressee",
    ]

    kept_lines = []

    for line in lines:
        lowered = line.lower()

        if any(
            lowered.startswith(prefix)
            for prefix in drop_prefixes
        ):
            continue

        if (
            arabic_ratio(line) < 0.10
            and latin_ratio(line) > 0.35
        ):
            continue

        kept_lines.append(line)

    cleaned = " ".join(
        kept_lines
    ).strip()

    if (
        not cleaned
        and arabic_ratio(text) > 0.10
    ):
        cleaned = " ".join([
            line
            for line in lines
            if arabic_ratio(line) > 0.10
        ]).strip()

    return cleaned


def load_system92_checkpoint(
    checkpoint_step,
):
    checkpoint_step = int(
        checkpoint_step
    )

    checkpoint_path = (
        CHECKPOINT_PATHS[
            checkpoint_step
        ]
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    dtype = (
        torch.bfloat16
        if (
            torch.cuda.is_available()
            and torch.cuda.is_bf16_supported()
        )
        else torch.float16
    )

    tokenizer = (
        AutoTokenizer.from_pretrained(
            str(BASE_MODEL_DIR),
            trust_remote_code=True,
            local_files_only=True,
            use_fast=True,

            # Required for the current
            # Transformers/tokenizer combination.
            extra_special_tokens={},
        )
    )

    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = (
            tokenizer.eos_token
        )

    base_model = (
        AutoModelForCausalLM
        .from_pretrained(
            str(BASE_MODEL_DIR),
            dtype=dtype,
            device_map="auto",
            trust_remote_code=True,
            local_files_only=True,
            attn_implementation="sdpa",
            low_cpu_mem_usage=True,
        )
    )

    model = PeftModel.from_pretrained(
        base_model,
        str(checkpoint_path),
        is_trainable=False,
    )

    model.config.use_cache = True
    model.config.pad_token_id = (
        tokenizer.pad_token_id
    )

    model.eval()

    return model, tokenizer


def generate_batch_safe(
    model,
    tokenizer,
    row_dicts,
    spec,
):
    prompts = [
        format_generation_prompt(
            row,
            spec,
        )
        for row in row_dicts
    ]

    try:
        encoded = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
        )

        device = (
            model
            .get_input_embeddings()
            .weight
            .device
        )

        encoded = {
            key: tensor.to(device)
            for key, tensor
            in encoded.items()
        }

        prompt_length = (
            encoded["input_ids"]
            .shape[-1]
        )

        with torch.inference_mode():
            generated = model.generate(
                **encoded,
                max_new_tokens=(
                    MAX_NEW_TOKENS
                ),
                eos_token_id=(
                    tokenizer.eos_token_id
                ),
                pad_token_id=(
                    tokenizer.pad_token_id
                ),
                **GENERATION_KWARGS,
            )

        results = []

        for row_position in range(
            len(row_dicts)
        ):
            generated_ids = generated[
                row_position,
                prompt_length:,
            ]

            raw_prediction = (
                tokenizer.decode(
                    generated_ids,
                    skip_special_tokens=False,
                )
            )

            raw_prediction = (
                strip_special_tokens(
                    raw_prediction,
                    tokenizer,
                )
            )

            clean_prediction = (
                postprocess_prediction(
                    raw_prediction,
                    tokenizer,
                )
            )

            final_prediction = (
                clean_prediction
                if clean_prediction
                else raw_prediction
            ).strip()

            error = (
                ""
                if final_prediction
                else (
                    "empty_prediction_"
                    "after_cleaning"
                )
            )

            results.append({
                "raw_prediction": (
                    raw_prediction
                ),
                "clean_prediction": (
                    clean_prediction
                ),
                "prediction": (
                    final_prediction
                ),
                "generation_error": error,
            })

        return results

    except RuntimeError as error:
        if len(row_dicts) > 1:
            print(
                "Splitting failed batch:",
                repr(error)[:500],
            )

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            midpoint = (
                len(row_dicts) // 2
            )

            return (
                generate_batch_safe(
                    model,
                    tokenizer,
                    row_dicts[:midpoint],
                    spec,
                )
                + generate_batch_safe(
                    model,
                    tokenizer,
                    row_dicts[midpoint:],
                    spec,
                )
            )

        return [{
            "raw_prediction": "",
            "clean_prediction": "",
            "prediction": "",
            "generation_error": (
                f"{type(error).__name__}: "
                f"{str(error)}"
            ),
        }]


print("Generation functions are ready.")

Generation functions are ready.


### **Run or resume test generation**

In [8]:
# ============================================================
# Cell 8A — Reuse exact matches from the old public-test run
# ============================================================

OLD_OUTPUT_DIR = (
    PROJECT_DIR
    / "inference_variants"
    / "92_official_test_system92_router_v1"
)

OLD_PREDICTION_PATH = (
    OLD_OUTPUT_DIR
    / "turn_predictions.csv"
)

OLD_TEST_CACHE_PATH = (
    PROJECT_DIR
    / "inference_variants"
    / "_shared_cache"
    / "official_test_system92_v1"
    / "official_test_f2aa0479fd8a.pkl"
)

REUSED_PREDICTION_ROWS = {}


def exact_reuse_key(row):
    payload = {
        "config": scalar_text(
            row.get("config", "")
        ),
        "dialect": scalar_text(
            row.get("dialect", "")
        ),
        "domain": scalar_text(
            row.get("domain", "")
        ),
        "participants": scalar_text(
            row.get("participants", "")
        ),
        "speaker": scalar_text(
            row.get("speaker", "")
        ),
        "gender_direction": scalar_text(
            row.get("gender_direction", "")
        ),
        "previous_english_turns": to_plain(
            row.get(
                "previous_english_turns",
                []
            )
        ),
        "source_text": scalar_text(
            row.get("source_text", "")
        )
    }

    serialized = json.dumps(
        payload,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":")
    )

    return hashlib.sha256(
        serialized.encode("utf-8")
    ).hexdigest()


if not OLD_PREDICTION_PATH.exists():
    print(
        "Old predictions were not found:",
        OLD_PREDICTION_PATH
    )

elif not OLD_TEST_CACHE_PATH.exists():
    print(
        "Old test dataframe was not found:",
        OLD_TEST_CACHE_PATH
    )

else:
    old_test_df = pd.read_pickle(
        OLD_TEST_CACHE_PATH
    )

    old_prediction_df = pd.read_csv(
        OLD_PREDICTION_PATH
    )

    old_test_df["source_id"] = (
        old_test_df["source_id"]
        .astype(str)
    )

    old_prediction_df["source_id"] = (
        old_prediction_df["source_id"]
        .astype(str)
    )

    old_prediction_df["prediction"] = (
        old_prediction_df["prediction"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    old_prediction_map = {
        str(row["source_id"]): row
        for row in old_prediction_df.to_dict(
            "records"
        )
        if str(
            row.get("prediction", "")
        ).strip()
    }

    old_test_df["_reuse_key"] = [
        exact_reuse_key(row)
        for row in old_test_df.to_dict(
            "records"
        )
    ]

    inference_test_df["_reuse_key"] = [
        exact_reuse_key(row)
        for row in inference_test_df.to_dict(
            "records"
        )
    ]

    old_key_counts = (
        old_test_df["_reuse_key"]
        .value_counts()
    )

    new_key_counts = (
        inference_test_df["_reuse_key"]
        .value_counts()
    )

    # Only reuse keys that are unique in both datasets.
    old_unique_df = old_test_df[
        old_test_df["_reuse_key"]
        .map(old_key_counts)
        .eq(1)
    ].copy()

    old_row_by_key = {
        row["_reuse_key"]: row
        for row in old_unique_df.to_dict(
            "records"
        )
    }

    for current_row in inference_test_df.to_dict(
        "records"
    ):
        reuse_key = current_row[
            "_reuse_key"
        ]

        if new_key_counts.get(
            reuse_key,
            0
        ) != 1:
            continue

        if reuse_key not in old_row_by_key:
            continue

        old_row = old_row_by_key[
            reuse_key
        ]

        old_source_id = str(
            old_row["source_id"]
        )

        if old_source_id not in old_prediction_map:
            continue

        old_prediction = old_prediction_map[
            old_source_id
        ]

        source_id = str(
            current_row["source_id"]
        )

        REUSED_PREDICTION_ROWS[
            source_id
        ] = {
            "source_id": source_id,
            "config": current_row["config"],
            "country": current_row["country"],
            "conversation_id": current_row[
                "conversation_id"
            ],
            "turn_order": int(
                current_row["turn_order"]
            ),
            "dialect": current_row["dialect"],
            "domain": current_row["domain"],
            "source_text": current_row[
                "source_text"
            ],
            "prediction": old_prediction[
                "prediction"
            ],
            "raw_prediction": old_prediction.get(
                "raw_prediction",
                ""
            ),
            "clean_prediction": old_prediction.get(
                "clean_prediction",
                old_prediction["prediction"]
            ),
            "generation_error": "",
            "route_checkpoint": int(
                current_row["route_checkpoint"]
            ),
            "route_variant": current_row[
                "route_variant"
            ],
            "route_selection_source": current_row[
                "route_selection_source"
            ],
            "retrieval_configs": ",".join(
                current_row[
                    "route_retrieval_configs"
                ]
            ),
            "use_participants": bool(
                current_row[
                    "route_use_participants"
                ]
            ),
            "decode_tag": DECODE_TAG,
            "prompt_version": PROMPT_VERSION,
            "reused_from_old_test": True,
            "old_source_id": old_source_id,
            "generated_at": time.strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        }

print()
print(
    "Old test turns:",
    (
        len(old_test_df)
        if "old_test_df" in globals()
        else 0
    )
)
print(
    "Old predictions:",
    (
        len(old_prediction_df)
        if "old_prediction_df" in globals()
        else 0
    )
)
print(
    "Safely reusable predictions:",
    len(REUSED_PREDICTION_ROWS),
    "/",
    len(inference_test_df)
)


Old test turns: 14442
Old predictions: 14442
Safely reusable predictions: 0 / 14459


In [10]:
# ============================================================
# Cell 9 — Run or resume official private-test generation
# ============================================================

if "REUSED_PREDICTION_ROWS" not in globals():
    raise RuntimeError(
        "Run Cell 8A before Cell 9."
    )

RUN_CONFIGURATION = {
    "run_name": OUTPUT_DIR.name,
    "dataset_name": DATASET_NAME,
    "dataset_revision": DATASET_REVISION,
    "test_id_hash": TEST_ID_HASH,
    "test_rows": int(
        len(inference_test_df)
    ),
    "system92_routes": ALL_ROUTES,
    "checkpoint_paths": {
        str(step): str(path)
        for step, path
        in CHECKPOINT_PATHS.items()
    },
    "base_model": str(
        BASE_MODEL_DIR
    ),
    "prompt_version": PROMPT_VERSION,
    "shot_fingerprint": SHOT_FINGERPRINT,
    "generation_kwargs": GENERATION_KWARGS,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "save_every": SAVE_EVERY
}

RUN_FINGERPRINT = hashlib.sha256(
    json.dumps(
        RUN_CONFIGURATION,
        ensure_ascii=False,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()

RUN_MANIFEST_PATH = (
    OUTPUT_DIR
    / "test_run_manifest.json"
)

if RUN_MANIFEST_PATH.exists():
    with open(
        RUN_MANIFEST_PATH,
        encoding="utf-8"
    ) as file:
        existing_manifest = json.load(
            file
        )

    if (
        existing_manifest.get(
            "run_fingerprint"
        )
        != RUN_FINGERPRINT
    ):
        raise RuntimeError(
            "The output directory contains a "
            "different private-test run."
        )

else:
    atomic_json_save(
        {
            **RUN_CONFIGURATION,
            "run_fingerprint": (
                RUN_FINGERPRINT
            ),
            "created_at": time.strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        },
        RUN_MANIFEST_PATH
    )

PREDICTION_PATH = (
    OUTPUT_DIR
    / "turn_predictions.csv"
)

expected_ids = set(
    inference_test_df[
        "source_id"
    ].astype(str)
)

order_df = inference_test_df[
    [
        "source_id",
        "_test_row_idx"
    ]
].copy()


def atomic_save_predictions(
    prediction_rows_by_id
):
    frame = pd.DataFrame(
        list(
            prediction_rows_by_id.values()
        )
    )

    if len(frame):
        frame["source_id"] = (
            frame["source_id"]
            .astype(str)
        )

        frame = (
            frame
            .merge(
                order_df,
                on="source_id",
                how="left",
                validate="one_to_one"
            )
            .sort_values(
                "_test_row_idx"
            )
            .drop(
                columns=[
                    "_test_row_idx"
                ]
            )
            .reset_index(drop=True)
        )

    temporary_path = Path(
        str(PREDICTION_PATH) + ".tmp"
    )

    frame.to_csv(
        temporary_path,
        index=False,
        encoding="utf-8-sig"
    )

    os.replace(
        temporary_path,
        PREDICTION_PATH
    )

    return frame


def create_prediction_record(
    source_row,
    result,
    checkpoint_step,
    route_variant,
    spec
):
    return {
        "source_id": str(
            source_row["source_id"]
        ),
        "config": source_row["config"],
        "country": source_row["country"],
        "conversation_id": source_row[
            "conversation_id"
        ],
        "turn_order": int(
            source_row["turn_order"]
        ),
        "dialect": source_row["dialect"],
        "domain": source_row["domain"],
        "source_text": source_row[
            "source_text"
        ],
        "prediction": result[
            "prediction"
        ],
        "raw_prediction": result[
            "raw_prediction"
        ],
        "clean_prediction": result[
            "clean_prediction"
        ],
        "generation_error": result[
            "generation_error"
        ],
        "route_checkpoint": int(
            checkpoint_step
        ),
        "route_variant": route_variant,
        "route_selection_source": source_row[
            "route_selection_source"
        ],
        "retrieval_configs": ",".join(
            source_row[
                "route_retrieval_configs"
            ]
        ),
        "use_participants": spec[
            "use_participants"
        ],
        "decode_tag": DECODE_TAG,
        "prompt_version": PROMPT_VERSION,
        "run_fingerprint": RUN_FINGERPRINT,
        "generated_at": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    }


# Seed exact matches from the old run.
prediction_rows_by_id = {
    source_id: {
        **record,
        "run_fingerprint": RUN_FINGERPRINT
    }
    for source_id, record
    in REUSED_PREDICTION_ROWS.items()
}

if prediction_rows_by_id:
    print(
        "Seeded reusable predictions:",
        len(prediction_rows_by_id)
    )

# Resume current private-test predictions.
if PREDICTION_PATH.exists():
    existing_df = pd.read_csv(
        PREDICTION_PATH
    )

    required_columns = {
        "source_id",
        "prediction",
        "generation_error",
        "run_fingerprint"
    }

    missing_columns = (
        required_columns
        - set(existing_df.columns)
    )

    if missing_columns:
        raise RuntimeError(
            "Invalid current prediction CSV. "
            f"Missing columns: "
            f"{sorted(missing_columns)}"
        )

    existing_df["source_id"] = (
        existing_df["source_id"]
        .astype(str)
    )

    existing_df = (
        existing_df[
            existing_df["source_id"]
            .isin(expected_ids)
        ]
        .drop_duplicates(
            subset=["source_id"],
            keep="last"
        )
        .copy()
    )

    fingerprints = set(
        existing_df[
            "run_fingerprint"
        ]
        .dropna()
        .astype(str)
    )

    if (
        fingerprints
        and fingerprints
        != {RUN_FINGERPRINT}
    ):
        raise RuntimeError(
            "Existing predictions have a "
            "different run fingerprint."
        )

    valid_mask = (
        existing_df["prediction"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )

    valid_mask &= (
        existing_df[
            "generation_error"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )

    valid_existing_df = (
        existing_df[
            valid_mask
        ].copy()
    )

    # Current private-test predictions take
    # precedence over recovered predictions.
    prediction_rows_by_id.update({
        str(record["source_id"]): record
        for record
        in valid_existing_df.to_dict(
            "records"
        )
    })

    print(
        "Resumed current-run predictions:",
        len(valid_existing_df)
    )

done_ids = set(
    prediction_rows_by_id
)

print(
    "Complete predictions:",
    len(done_ids),
    "/",
    len(inference_test_df)
)

print(
    "Missing predictions:",
    len(expected_ids - done_ids)
)

generated_since_save = 0

for checkpoint_step in sorted(
    inference_test_df[
        "route_checkpoint"
    ].unique()
):
    checkpoint_step = int(
        checkpoint_step
    )

    checkpoint_df = (
        inference_test_df[
            inference_test_df[
                "route_checkpoint"
            ]
            == checkpoint_step
        ]
        .copy()
    )

    checkpoint_pending = (
        checkpoint_df[
            ~checkpoint_df[
                "source_id"
            ]
            .astype(str)
            .isin(done_ids)
        ]
        .copy()
    )

    if len(checkpoint_pending) == 0:
        print(
            f"Checkpoint {checkpoint_step}: "
            "already complete."
        )
        continue

    print()
    print("=" * 80)
    print(
        "LOADING CHECKPOINT:",
        checkpoint_step
    )
    print(
        "Pending:",
        len(checkpoint_pending)
    )
    print("=" * 80)

    model = None
    tokenizer = None

    try:
        model, tokenizer = (
            load_system92_checkpoint(
                checkpoint_step
            )
        )

        for (
            route_variant,
            route_df
        ) in checkpoint_pending.groupby(
            "route_variant",
            sort=True
        ):
            route_df = (
                route_df
                .sort_values(
                    "_test_row_idx"
                )
                .reset_index(drop=True)
            )

            first_row = route_df.iloc[0]

            spec = {
                "use_direction": True,
                "use_previous_speakers": True,
                "use_participants": bool(
                    first_row[
                        "route_use_participants"
                    ]
                )
            }

            print()
            print(
                "Route:",
                route_variant
            )
            print(
                "Rows:",
                len(route_df)
            )
            print(
                "Participants:",
                spec["use_participants"]
            )

            pending_batch = []

            for _, row in tqdm(
                route_df.iterrows(),
                total=len(route_df),
                desc=(
                    f"{checkpoint_step}:"
                    f"{route_variant}"
                )
            ):
                source_row = row.to_dict()

                source_id = str(
                    source_row["source_id"]
                )

                if source_id in done_ids:
                    continue

                pending_batch.append(
                    source_row
                )

                if (
                    len(pending_batch)
                    < GEN_BATCH_SIZE
                ):
                    continue

                batch_results = (
                    generate_batch_safe(
                        model,
                        tokenizer,
                        pending_batch,
                        spec
                    )
                )

                for (
                    one_row,
                    result
                ) in zip(
                    pending_batch,
                    batch_results
                ):
                    source_id = str(
                        one_row["source_id"]
                    )

                    prediction_rows_by_id[
                        source_id
                    ] = create_prediction_record(
                        one_row,
                        result,
                        checkpoint_step,
                        route_variant,
                        spec
                    )

                    if not result[
                        "generation_error"
                    ]:
                        done_ids.add(
                            source_id
                        )

                    generated_since_save += 1

                pending_batch = []

                if (
                    generated_since_save
                    >= SAVE_EVERY
                ):
                    saved_df = (
                        atomic_save_predictions(
                            prediction_rows_by_id
                        )
                    )

                    print(
                        "Saved:",
                        len(saved_df),
                        "/",
                        len(inference_test_df)
                    )

                    generated_since_save = 0

            if pending_batch:
                batch_results = (
                    generate_batch_safe(
                        model,
                        tokenizer,
                        pending_batch,
                        spec
                    )
                )

                for (
                    one_row,
                    result
                ) in zip(
                    pending_batch,
                    batch_results
                ):
                    source_id = str(
                        one_row["source_id"]
                    )

                    prediction_rows_by_id[
                        source_id
                    ] = create_prediction_record(
                        one_row,
                        result,
                        checkpoint_step,
                        route_variant,
                        spec
                    )

                    if not result[
                        "generation_error"
                    ]:
                        done_ids.add(
                            source_id
                        )

                    generated_since_save += 1

                if (
                    generated_since_save
                    >= SAVE_EVERY
                ):
                    saved_df = (
                        atomic_save_predictions(
                            prediction_rows_by_id
                        )
                    )

                    print(
                        "Saved:",
                        len(saved_df),
                        "/",
                        len(inference_test_df)
                    )

                    generated_since_save = 0

    finally:
        if prediction_rows_by_id:
            atomic_save_predictions(
                prediction_rows_by_id
            )

        if model is not None:
            del model

        if tokenizer is not None:
            del tokenizer

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

            if hasattr(
                torch.cuda,
                "ipc_collect"
            ):
                torch.cuda.ipc_collect()

# Also saves when everything came from reuse/resume.
if prediction_rows_by_id:
    atomic_save_predictions(
        prediction_rows_by_id
    )

if not PREDICTION_PATH.exists():
    raise RuntimeError(
        "No private-test prediction CSV was created."
    )

final_test_predictions_df = pd.read_csv(
    PREDICTION_PATH
)

final_test_predictions_df["source_id"] = (
    final_test_predictions_df[
        "source_id"
    ].astype(str)
)

final_test_predictions_df["prediction"] = (
    final_test_predictions_df[
        "prediction"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)

final_test_predictions_df[
    "generation_error"
] = (
    final_test_predictions_df[
        "generation_error"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)

actual_ids = set(
    final_test_predictions_df[
        "source_id"
    ]
)

missing_ids = (
    expected_ids
    - actual_ids
)

extra_ids = (
    actual_ids
    - expected_ids
)

duplicate_count = int(
    final_test_predictions_df[
        "source_id"
    ].duplicated().sum()
)

empty_count = int(
    final_test_predictions_df[
        "prediction"
    ].eq("").sum()
)

error_count = int(
    final_test_predictions_df[
        "generation_error"
    ].ne("").sum()
)

print()
print("=" * 80)
print(
    "FINAL PRIVATE-TEST "
    "GENERATION VALIDATION"
)
print("=" * 80)
print(
    "Rows:",
    len(final_test_predictions_df)
)
print(
    "Expected:",
    len(inference_test_df)
)
print(
    "Missing:",
    len(missing_ids)
)
print(
    "Extra:",
    len(extra_ids)
)
print(
    "Duplicates:",
    duplicate_count
)
print(
    "Empty:",
    empty_count
)
print(
    "Errors:",
    error_count
)

if error_count:
    display(
        final_test_predictions_df[
            final_test_predictions_df[
                "generation_error"
            ].ne("")
        ][
            [
                "source_id",
                "source_text",
                "generation_error"
            ]
        ].head(20)
    )

if (
    missing_ids
    or extra_ids
    or duplicate_count
    or empty_count
    or error_count
    or len(final_test_predictions_df)
    != len(inference_test_df)
):
    raise RuntimeError(
        "Private-test generation is incomplete. "
        "Rerun Cell 9 to retry missing/error rows."
    )

ROUTING_SUMMARY_PATH = (
    OUTPUT_DIR
    / "test_routing_summary.csv"
)

routing_summary = (
    final_test_predictions_df
    .groupby(
        [
            "config",
            "route_checkpoint",
            "route_variant",
            "route_selection_source",
            "retrieval_configs"
        ],
        as_index=False,
        dropna=False
    )
    .agg(
        conversations=(
            "conversation_id",
            "nunique"
        ),
        turns=(
            "source_id",
            "size"
        )
    )
)

routing_summary.to_csv(
    ROUTING_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print()
print(
    "Private-test generation complete."
)
display(routing_summary)

Resumed current-run predictions: 13509
Complete predictions: 13509 / 14459
Missing predictions: 950
Checkpoint 16000: already complete.
Checkpoint 16500: already complete.

LOADING CHECKPOINT: 16600
Pending: 950

Route: 05_retrieved_two_shot_with_participants
Rows: 35
Participants: True


16600:05_retrieved_two_shot_with_participants:   0%|          | 0/35 [00:00<?, ?it/s]


Route: 92_unseen_SD_proxy_EG
Rows: 915
Participants: True


16600:92_unseen_SD_proxy_EG:   0%|          | 0/915 [00:00<?, ?it/s]

Saved: 13610 / 14459
Saved: 13710 / 14459
Saved: 13810 / 14459
Saved: 13910 / 14459
Saved: 14010 / 14459
Saved: 14110 / 14459
Saved: 14210 / 14459
Saved: 14310 / 14459
Saved: 14410 / 14459

FINAL PRIVATE-TEST GENERATION VALIDATION
Rows: 14459
Expected: 14459
Missing: 0
Extra: 0
Duplicates: 0
Empty: 0
Errors: 0

Private-test generation complete.


,config,route_checkpoint,route_variant,route_selection_source,retrieval_configs,conversations,turns
0,EG,16600,05_retrieved_two_shot_with_participants,system92_dev,EG,359,1113
1,JO,16600,03_retrieved_two_shot,system92_dev,JO,348,1109
2,LB,16000,07_ckpt16000_retrieved_two_shot,system92_dev,LB,368,1110
3,LY,16000,92_unseen_LY_proxy_TN,unseen_geographic_proxy_TN,TN,423,1309
4,MA,16600,05_retrieved_two_shot_with_participants,system92_dev,MA,362,1111
5,MR,16000,07_ckpt16000_retrieved_two_shot,system92_dev,MR,365,1119
6,OM,16600,05_retrieved_two_shot_with_participants,system92_dev,OM,370,1107
7,PS,16500,06_ckpt16500_retrieved_two_shot,system92_dev,PS,351,1111
8,SA,16500,06_ckpt16500_retrieved_two_shot,system92_dev,SA,359,1114
9,SD,16600,92_unseen_SD_proxy_EG,unseen_geographic_proxy_EG,EG,283,915


### **Create and validate the official submission ZIP**

In [11]:
if not PREDICTION_PATH.exists():
    raise FileNotFoundError(
        f"Missing predictions: "
        f"{PREDICTION_PATH}"
    )

submission_turn_df = pd.read_csv(
    PREDICTION_PATH
)

submission_turn_df["source_id"] = (
    submission_turn_df[
        "source_id"
    ].astype(str)
)

submission_turn_df["config"] = (
    submission_turn_df[
        "config"
    ].astype(str)
)

submission_turn_df[
    "conversation_id"
] = (
    submission_turn_df[
        "conversation_id"
    ].astype(str)
)

submission_turn_df["turn_order"] = (
    pd.to_numeric(
        submission_turn_df[
            "turn_order"
        ],
        errors="raise",
    ).astype(int)
)

submission_turn_df["prediction"] = (
    submission_turn_df[
        "prediction"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)

expected_ids = set(
    official_test_df[
        "source_id"
    ].astype(str)
)

actual_ids = set(
    submission_turn_df[
        "source_id"
    ].astype(str)
)

if actual_ids != expected_ids:
    raise RuntimeError(
        "Prediction IDs do not exactly match "
        "the official test IDs."
    )

if submission_turn_df[
    "source_id"
].duplicated().any():
    raise RuntimeError(
        "Duplicate source_id in predictions."
    )

if (
    submission_turn_df[
        "prediction"
    ]
    == ""
).any():
    raise RuntimeError(
        "Empty predictions remain."
    )

submission_turn_df = (
    submission_turn_df
    .sort_values([
        "config",
        "conversation_id",
        "turn_order",
    ])
    .reset_index(drop=True)
)

submission_records = []

for (
    config_name,
    conversation_id,
), conversation_df in (
    submission_turn_df.groupby(
        [
            "config",
            "conversation_id",
        ],
        sort=True,
    )
):
    conversation_df = (
        conversation_df
        .sort_values("turn_order")
    )

    if conversation_df[
        "turn_order"
    ].duplicated().any():
        raise RuntimeError(
            "Duplicate turn_order inside "
            f"{config_name}/"
            f"{conversation_id}"
        )

    turns = [
        {
            "turn_order": int(
                row.turn_order
            ),
            "prediction": str(
                row.prediction
            ),
        }
        for row
        in conversation_df.itertuples()
    ]

    submission_records.append({
        "conv_id": str(
            conversation_id
        ),
        "country": str(
            config_name
        ),
        "turns": turns,
    })

submission_turn_count = sum(
    len(record["turns"])
    for record in submission_records
)

EXPECTED_TEST_TURNS = len(
    official_test_df
)

if (
    submission_turn_count
    != EXPECTED_TEST_TURNS
):
    raise RuntimeError(
        "Submission turn-count mismatch: "
        f"{submission_turn_count} "
        f"!= {EXPECTED_TEST_TURNS}"
    )

JSONL_PATH = (
    OUTPUT_DIR
    / "predictions.jsonl"
)

ZIP_PATH = (
    OUTPUT_DIR
    / "submission_predictions.zip"
)

temporary_jsonl_path = Path(
    str(JSONL_PATH) + ".tmp"
)

with open(
    temporary_jsonl_path,
    "w",
    encoding="utf-8",
) as file:
    for record in submission_records:
        file.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )

os.replace(
    temporary_jsonl_path,
    JSONL_PATH,
)

temporary_zip_path = Path(
    str(ZIP_PATH) + ".tmp"
)

with zipfile.ZipFile(
    temporary_zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        JSONL_PATH,
        arcname="predictions.jsonl",
    )

os.replace(
    temporary_zip_path,
    ZIP_PATH,
)


# ============================================================
# Exact ZIP readback validation
# ============================================================

readback_keys = set()
readback_turn_count = 0
readback_conversations = 0

with zipfile.ZipFile(
    ZIP_PATH,
    "r",
) as archive:
    if archive.namelist() != [
        "predictions.jsonl"
    ]:
        raise RuntimeError(
            "ZIP must contain only "
            "predictions.jsonl."
        )

    with archive.open(
        "predictions.jsonl",
        "r",
    ) as file:
        for binary_line in file:
            record = json.loads(
                binary_line.decode(
                    "utf-8"
                )
            )

            readback_conversations += 1

            config_name = str(
                record["country"]
            )

            conversation_id = str(
                record["conv_id"]
            )

            for turn in record["turns"]:
                readback_turn_count += 1

                readback_keys.add((
                    config_name,
                    conversation_id,
                    int(
                        turn[
                            "turn_order"
                        ]
                    ),
                ))

expected_keys = set(zip(
    official_test_df[
        "config"
    ].astype(str),
    official_test_df[
        "conversation_id"
    ].astype(str),
    official_test_df[
        "turn_order"
    ].astype(int),
))

if readback_turn_count != EXPECTED_TEST_TURNS:
    raise RuntimeError(
        "ZIP readback turn-count mismatch."
    )

if readback_keys != expected_keys:
    raise RuntimeError(
        "ZIP readback keys do not match "
        "the official test set."
    )

submission_manifest = {
    "system": (
        "92_mixed_best_checkpoint_"
        "variant_per_country"
    ),
    "run_fingerprint": (
        RUN_FINGERPRINT
    ),
    "dataset_revision": (
        DATASET_REVISION
    ),
    "test_turns": int(
        EXPECTED_TEST_TURNS
    ),
    "conversations": int(
        readback_conversations
    ),
    "test_configs": TEST_CONFIGS,
    "unseen_test_configs": (
        UNSEEN_TEST_CONFIGS
    ),
    "unseen_fallbacks": (
        UNSEEN_ROUTES
    ),
    "jsonl_path": str(
        JSONL_PATH
    ),
    "zip_path": str(
        ZIP_PATH
    ),
    "completed_at": time.strftime(
        "%Y-%m-%d %H:%M:%S"
    ),
}

atomic_json_save(
    submission_manifest,
    OUTPUT_DIR
    / "submission_manifest.json",
)

print("\n" + "=" * 90)
print("OFFICIAL TEST SUBMISSION READY")
print("=" * 90)
print("System: System 92")
print("Turns:", EXPECTED_TEST_TURNS)
print(
    "Conversations:",
    readback_conversations,
)
print("Configs:", TEST_CONFIGS)
print(
    "Zero-shot configs:",
    UNSEEN_TEST_CONFIGS,
)
print("\nSUBMIT THIS ZIP:")
print(ZIP_PATH)

display(
    routing_summary
)

display(
    FileLink(
        str(ZIP_PATH)
    )
)


OFFICIAL TEST SUBMISSION READY
System: System 92
Turns: 14459
Conversations: 4673
Configs: ['EG', 'JO', 'LB', 'LY', 'MA', 'MR', 'OM', 'PS', 'SA', 'SD', 'SY', 'TN', 'YE']
Zero-shot configs: ['LY', 'SD']

SUBMIT THIS ZIP:
/home/mabdallah/alexandriax_mt_14d/inference_variants/92_official_private_test_system92_router_v2/submission_predictions.zip


,config,route_checkpoint,route_variant,route_selection_source,retrieval_configs,conversations,turns
0,EG,16600,05_retrieved_two_shot_with_participants,system92_dev,EG,359,1113
1,JO,16600,03_retrieved_two_shot,system92_dev,JO,348,1109
2,LB,16000,07_ckpt16000_retrieved_two_shot,system92_dev,LB,368,1110
3,LY,16000,92_unseen_LY_proxy_TN,unseen_geographic_proxy_TN,TN,423,1309
4,MA,16600,05_retrieved_two_shot_with_participants,system92_dev,MA,362,1111
5,MR,16000,07_ckpt16000_retrieved_two_shot,system92_dev,MR,365,1119
6,OM,16600,05_retrieved_two_shot_with_participants,system92_dev,OM,370,1107
7,PS,16500,06_ckpt16500_retrieved_two_shot,system92_dev,PS,351,1111
8,SA,16500,06_ckpt16500_retrieved_two_shot,system92_dev,SA,359,1114
9,SD,16600,92_unseen_SD_proxy_EG,unseen_geographic_proxy_EG,EG,283,915


/home/mabdallah/alexandriax_mt_14d/inference_variants/92_official_private_test_system92_router_v2/submission_predictions.zip

### **New Experiment**